## Session 2, in class: value function iteration on a grid (gap version)

Computational Macro (HWS 2026), University of Mannheim

The deterministic growth model of session 1, solved by value function iteration
on a grid. Every gap is marked `___` and corresponds to one line of the
pseudo-code on the slides. Fill the gaps from top to bottom and run the cells
with Shift+Enter. The last section is for experimenting once everything runs.

**The problem** (session 1, "The same example by dynamic programming"):

$$
v(k) = \max_{0 \leq k' \leq k^\alpha} \bigl\{ \ln(k^\alpha - k') + \beta \, v(k') \bigr\}
$$

with the closed form as the benchmark for everything we compute:

$$
v(k) = E + F \ln k, \qquad F = \frac{\alpha}{1 - \alpha\beta}, \qquad
k' = \alpha\beta \, k^\alpha, \qquad k^* = (\alpha\beta)^{\frac{1}{1-\alpha}}.
$$

**Notation, slides to code**

| Slides | Meaning | Code |
| --- | --- | --- |
| $N, \varepsilon, \overline{n}$ | grid size, tolerance, iteration cap | `mpar.nk`, `mpar.crit`, `mpar.maxiter` |
| $\alpha, \beta$ | capital share, discount factor | `par.α`, `par.β` |
| $\mathcal{K} = \{k_1, \ldots, k_N\}$ | the grid | `gri.k` |
| $k_i, k'_j$ on the table | state in row $i$, choice in column $j$ | `meshes.k`, `meshes.kprime` |
| $\mathbf{U}$ | reward matrix, $-\infty$ where infeasible | `U` |
| $\mathbf{M}(\mathbf{v}) = \mathbf{U} + \beta \mathbf{1} \mathbf{v}^\top$ | right hand side for every pair | `M` |
| $\mathbf{v}$, $\mathbf{v}^{\text{new}}$ | value function on the grid, a vector | `v`, `vnew` |
| $\mathbf{g}$, $k_{g_i}$ | policy as grid indices, and in levels | `g`, `kprime` |
| $d_n = \lVert \mathbf{v}^{\text{new}} - \mathbf{v} \rVert_\infty$ | the stopping signal | `dd` |

Julia and the slides both count from 1.

### 0. Packages

Plotting and formatted printing. Nothing else is needed today.

In [ ]:
using Plots
using Printf

### 1. Parameters: economics

One container for the economics, one for the numerics (week 1 primer, part 3).
Keeping them apart makes the experiments at the end safe: when you change the
grid, you can see at a glance that no economics moved.

The economic parameters are the capital share $\alpha$ and the discount factor
$\beta$. Utility is $\ln c$, the case with a closed form.

In [ ]:
Base.@kwdef struct EconomicParameters
    α::Float64 = 0.3          # capital share
    β::Float64 = 0.96         # discount factor
end

### 1. Parameters: numerics

Everything that belongs to the *method* and not to the *model*: the number of
grid points $N$, the grid bounds relative to the steady state $k^*$, the spacing
of the points, the tolerance $\varepsilon$, and the iteration cap $\overline{n}$.

In [ ]:
Base.@kwdef struct NumericalParameters
    nk::Int = 200             # number of points on the capital grid, N
    kmin_rel::Float64 = 0.1   # lower grid bound, relative to k*
    kmax_rel::Float64 = 2.0   # upper grid bound, relative to k*
    spacing::Symbol = :linear # :linear or :log
    crit::Float64 = 1e-6      # convergence tolerance, ε
    maxiter::Int = 5000       # iteration cap, the safety net
end

Create both containers with their defaults and print them. Every result you
report should come with these two lines, so that a reader knows what was run.

In [ ]:
par = EconomicParameters()
mpar = NumericalParameters()
println("Economic parameters:  ", par)
println("Numerical parameters: ", mpar)

### 2. The grid: bounds

The grid has to contain the region the household actually visits. For the growth
model that is a neighbourhood of the steady state of the closed form policy,

$$
k^* = (\alpha\beta)^{\frac{1}{1-\alpha}},
$$

so the bounds are set relative to $k^*$.

In [ ]:
kstar = ___                                 # k* = (αβ)^(1/(1-α)), fields are par.α and par.β, powers use ^
kmin, kmax = mpar.kmin_rel * kstar, mpar.kmax_rel * kstar
@printf("k* = %.4f, grid bounds %.4f to %.4f\n", kstar, kmin, kmax)

### 2. The grid: spacing

Two ways to place $N$ points between the bounds. **Equidistant** in $k$, or
**equidistant in $\ln k$** ("log spaced"), which puts more points at low $k$ where
the value function is most curved. Both are set by `mpar.spacing`.

The grid is stored in a container `gri`, so that later files can hold several
grids side by side.

In [ ]:
if mpar.spacing == :linear
    gri = (k = collect(range(kmin, kmax, length = mpar.nk)),)
else                                         # equidistant in ln k
    gri = (k = exp.(range(___, ___, length = mpar.nk)),) # equidistant in ln k: the range runs from the log of the lower bound to the log of the upper bound
end
@printf("%d points, first spacing %.5f, last spacing %.5f\n", mpar.nk, gri.k[2] - gri.k[1], gri.k[end] - gri.k[end-1])

### 3. The utility function

$u(c) = \ln c$, the log utility of session 1, for which the closed form exists.
Problem set 1 generalizes this function and the resource constraint.

In [ ]:
util(c) = ___                               # the natural logarithm of c
util(1.0)

### 4. The table: meshes

The slides organise the problem as an $N \times N$ table, one row per state $k_i$
and one column per choice $k'_j$. In code that table is two matrices of the same
shape: one holds $k_i$ in every entry of row $i$, the other holds $k'_j$ in every
entry of column $j$. Everything that follows is elementwise arithmetic on these.

In [ ]:
meshes = (
    k      = [k  for k in gri.k, kP in gri.k],   # k_i in row i, the same in every column
    kprime = [kP for k in gri.k, kP in gri.k],   # k'_j in column j, the same in every row
)
size(meshes.k), size(meshes.kprime)

### 4. The table: resources and consumption

At state $k_i$ the household has output $k_i^\alpha$ to spend. Choosing $k'_j$
leaves consumption

$$
C_{ij} = k_i^\alpha - k'_j .
$$

A pair with $C_{ij} \leq 0$ is infeasible.

In [ ]:
Y = ___                                     # output k_i^α at every state: the mesh of k_i to the power α, elementwise with .^
C = ___                                     # resources minus the choice: Y minus the mesh of k'_j, elementwise with .-
println("share of feasible pairs: ", round(count(C .> 0) / length(C), digits = 4))

### 4. The table: the reward matrix $\mathbf{U}$

$$
U_{ij} = \begin{cases} u(C_{ij}) & \text{if } C_{ij} > 0, \\ -\infty & \text{otherwise.} \end{cases}
$$

Setting infeasible pairs to $-\infty$ is not a trick. It is the correct value of a
choice outside the feasible set, and it makes the constraint disappear from the
rest of the code: the maximization can never pick such a pair. $\mathbf{U}$ does
not depend on $\mathbf{v}$, so it is built **once**, before the loop.

In [ ]:
U = fill(-Inf, size(C))
U[C .> 0] = ___                             # utility of the feasible consumption levels C[C .> 0], broadcast util with a dot
println("no NaN in U: ", !any(isnan, U))

### 4. The benchmark

The closed form on the grid, to check against later: $v(k_i) = E + F \ln k_i$
with $F = \alpha / (1 - \alpha\beta)$, and the policy $k' = \alpha\beta k_i^\alpha$.
The constant $E$ was derived in part 3 of the week 1 primer.

In [ ]:
F_star = par.α / (1 - par.α * par.β)
E_star = (log(1 - par.α * par.β) + par.β * F_star * log(par.α * par.β)) / (1 - par.β)
v_true = E_star .+ F_star .* log.(gri.k)
kprime_true = par.α * par.β .* gri.k .^ par.α
@printf("E = %.4f, F = %.4f\n", E_star, F_star)

### 4. The loop

The pseudo-code from the slides, line by line:

```
in    grid K, parameters, tolerance ε, iteration cap nbar
1     build U once
2     v <- 0,  n <- 0,  d <- ∞
3     while d > ε and n < nbar
4          M <- U + β 1 v'
5          v_new[i] <- max_j M[i, j],   g[i] <- argmax_j M[i, j]
6          d <- ‖v_new - v‖_∞
7          v <- v_new,  n <- n + 1
8     end while
out   v, g, n, and the certificate β / (1 - β) d
```

Line 4 is one broadcast: $\beta v_j$ depends on the column only, so the row vector
$\beta \mathbf{v}^\top$ is added to every row of $\mathbf{U}$. Line 5 takes the
maximum **along each row** and remembers where it sits.

In [ ]:
timer = time()
v      = zeros(mpar.nk)                      # line 2: the guess v_0 = 0
g      = ones(Int, mpar.nk)                  # the policy as grid indices
distVF = [Inf]                               # distances d_n, one per iteration
iterVF = 0
while ___ && iterVF < mpar.maxiter          # line 3: keep going while the last distance, distVF[end], exceeds mpar.crit
    global v, g, iterVF                      # the loop sits at the top level of the notebook
    M    = ___                              # line 4: U plus β times the row vector v', the dots .+ and .* broadcast it over every row
    vals, idx = findmax(M, dims = 2)         # line 5: maximum along each row ...
    vnew = vec(vals)
    g    = ___                              # idx holds a CartesianIndex (i, j) per row: getindex.(idx, 2) picks the column j, vec turns it into a vector
    dd   = ___                              # line 6: the max norm of vnew minus v, maximum(abs.( ... ))
    v    = vnew                              # line 7: update
    iterVF += 1
    push!(distVF, dd)
end
time_vfi = time() - timer
@printf("iterations: %d, final distance: %.3e, time: %.3f s\n", iterVF, distVF[end], time_vfi)

### 4. Did it converge?

A loop that hits the iteration cap has **not** converged, and the code must say
so loudly rather than hand back whatever it had. Then translate the policy from
grid indices into levels, $k_{g_i}$.

In [ ]:
if distVF[end] > mpar.crit
    error("not converged: the iteration cap was hit")
end
kprime = gri.k[g]                            # the policy in levels, k_{g_i}
kprime[1:5]

### 5. Check 1: the boundary

If the policy ever sits on the first or last grid point, the grid does not
contain the region the household visits, and every number that follows is
meaningless. One line, and it catches a real bug.

In [ ]:
println("policy at the lowest grid point:  ", count(g .== 1), " states")
println("policy at the highest grid point: ", count(g .== mpar.nk), " states")

### 5. Checks 2 and 3: against the closed form

The grid policy can only take the $N$ values on the grid, so the best it can do
is land within half a grid spacing of $\alpha\beta k^\alpha$. The value error
mixes the grid error with the stopping error, and the loop certifies the latter:

$$
\lVert \mathbf{v}_n - \mathbf{v}_{\mathcal{K}} \rVert_\infty \leq \frac{\beta}{1-\beta} \, d_n .
$$

In [ ]:
@printf("largest policy error: %.2e   (first grid spacing: %.2e)\n", maximum(abs.(kprime .- kprime_true)), gri.k[2] - gri.k[1])
@printf("largest value error:  %.2e   (certificate β/(1-β) d: %.2e)\n", maximum(abs.(v .- v_true)), par.β / (1 - par.β) * distVF[end])

### 5. Check 4: the contraction rate

The distance between successive iterates should fall by the factor $\beta$ per
iteration. The ratio $d_n / d_{n-1}$ is the cheapest diagnostic you will ever
write. Print it in every value function iteration you code.

In [ ]:
@printf("last ratio d_n / d_(n-1): %.4f   (β = %.2f)\n", distVF[end] / distVF[end-1], par.β)

### 5. Plots

Value and policy against the closed form, and the distance per iteration on a
log scale, where geometric convergence is a straight line.

In [ ]:
p1 = plot(gri.k, v, linewidth = 2, label = "VFI on the grid", legend = :bottomright,
          xlabel = "capital k", title = "Value function")
plot!(p1, gri.k, v_true, color = :black, linestyle = :dash, label = "closed form E + F ln k")

p2 = plot(gri.k, kprime, seriestype = :stepmid, linewidth = 1.5, label = "VFI on the grid", legend = :topleft,
          xlabel = "capital today k", ylabel = "capital tomorrow k'", title = "Policy function")
plot!(p2, gri.k, kprime_true, color = :black, linestyle = :dash, label = "closed form αβ k^α")
plot!(p2, gri.k, gri.k, color = :gray, linewidth = 0.8, label = "45 degree line")

p3 = plot(1:iterVF, distVF[2:end], yaxis = :log10, linewidth = 2, label = "distance d_n",
          xlabel = "iteration n", title = "Distance between iterates")
hline!(p3, [mpar.crit], color = :gray, linestyle = :dot, label = "tolerance")

plot(p1, p2, p3, layout = (1, 3), size = (1400, 400), margin = 5Plots.mm)

### 6. Experiment: the grid is a modelling choice

Change one field of `mpar` at a time in the cell below, then rerun sections 2
to 5, and note what happens to the policy error, the iteration count, and the
boundary check.

| Try | Question |
| --- | --- |
| `mpar = NumericalParameters(nk = 30)`, then `nk = 2000` | fewer points, many points: which errors move? |
| `mpar = NumericalParameters(spacing = :log)` | more points at low $k$: does it help, and where? |
| `mpar = NumericalParameters(kmax_rel = 0.8)` | is the upper bound still large enough? |
| `mpar = NumericalParameters(kmin_rel = 0.5)` | and the lower bound? |
| `mpar = NumericalParameters(crit = 1e-10)` | does a tighter tolerance buy accuracy? |

Questions to answer in your pair:

1. Which of the three errors (policy, value, distance) responds to $N$, which to $\varepsilon$?
2. Does log spacing help here? Where along the grid does it help, where does it hurt?
3. What does the boundary check say when `kmax_rel` is $0.8$, and why is the result then unusable?
4. How does the iteration count depend on the grid? On $\beta$? Try $\beta = 0.99$.

In [ ]:
# your experiments: for example
# mpar = NumericalParameters(spacing = :log)
# then rerun the cells of sections 2 to 5